# BSM Reduced-Form Model: Quickstart

This notebook loads the released model, selects one output series from its
metadata, predicts five example scenarios, and reshapes the results by year.

In [ ]:
from pathlib import Path

import pandas as pd

from bsm_public_rf import load_model, output_catalog, unpack_outputs

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

## 1. Load the model and inputs

`example_inputs.csv` holds five synthetic scenarios drawn uniformly within each
input's historical range. They are illustrative, not meaningful BSM scenarios.
To use your own inputs, provide a CSV with one column per required input; see
`model.input_schema()` for names and available units and ranges.

In [ ]:
model = load_model()
inputs = pd.read_csv(ROOT / "examples" / "example_inputs.csv", index_col="scenario")

print(
    f"{len(model.required_input_names)} inputs -> {len(model.feature_names)} features "
    f"-> {len(model.output_names)} outputs"
)
model.input_schema().head()

## 2. Choose outputs and predict

`output_catalog` describes every output without requiring you to parse its name.
This filter selects one pathway and region across all available years. Omit the
`outputs` argument to `model.predict` when you need the complete output set.

In [ ]:
catalog = output_catalog(model.output_names)
output_names = catalog.query(
    "variable == 'AHC.MFSPMetric' and pathway == 'HEFA' and region == 'A'"
).index.tolist()
predictions = model.predict(inputs, outputs=output_names)
predictions.iloc[:, :5]

## 3. Understand the outputs

Output names follow `VARIABLE[pathway, region, product]_YEAR`. The catalog
splits each name and attaches its units and description from the bundled
metadata.

In [ ]:
catalog.drop_duplicates("variable")[["unit", "description"]]

## 4. Unpack into time series

`unpack_outputs` gives one row per scenario and selected series, with one column
per year. Region codes are expanded to full names. Use bracket
indexing (`df["sample"]`), because `sample` and `product` are also DataFrame methods.

In [ ]:
unpacked = unpack_outputs(predictions)
print(unpacked.shape)

unpacked[
    (unpacked["sample"] == "scenario_1")
    & (unpacked["variable"] == "AHC.MFSPMetric")
    & (unpacked["pathway"] == "HEFA")
    & (unpacked["region"] == "Atlantic")
]